# Transaction-monitoring assessment for Bank N
Data: IBM Transactions for AML HI-Medium (**PUBLIC DATA, synthetic**, AMLSim, CDLA-Sharing-1.0). Bank N layer (customers, KYC, rules, alerts, decisions): **SIMULATED**. Country tiers: **ILLUSTRATIVE**.

Order: data, quality, business meaning, rule validation, tuning, network, challenger, recommendations. Each step reads the outputs of the pipeline (`src/`), so the notebook stays short.

In [1]:
import json, pandas as pd, duckdb
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
O = ROOT / 'output'
f = json.loads((O / 'findings.json').read_text())
pd.set_option('display.width', 160, 'display.max_colwidth', 80)
def t(n, k=8): return pd.read_csv(O / f'{n}.csv').head(k)

## 1. Data: 32M transfers processed with Dask
CSV to Parquet partitioned by day; bank codes read as text; amounts converted to USD.

In [2]:
t('ingest_log')

,engine,dataset,rows_in_csv,rows_written,seconds,peak_rss_gb,cores,ram_gb
0,dask,HI-Small,5078345,5078345,15.9,3.08,12,16.8
1,duckdb,HI-Small,5078345,5078345,2.1,3.58,12,16.8
2,dask,HI-Medium,31898238,31898238,98.6,4.87,12,16.8


In [3]:
con = duckdb.connect()
con.sql(f"SELECT count(*) AS rows, sum(is_laundering) AS laundering, min(day) AS first_day, max(day) AS last_day "
        f"FROM read_parquet('{(ROOT / 'data/parquet/trans/**/*.parquet').as_posix()}', hive_partitioning=true)").df()

,rows,laundering,first_day,last_day
0,31898238,35230.0,2022-09-01,2022-09-28


## 2. Data quality: automated controls

In [4]:
t('controls_report', 15)[['control_id', 'name', 'status', 'detail']]

,control_id,name,status,detail
0,C01,Field completeness,WARN,worst field: to_bank (0.367%); all others 0.000% or less
1,C02,Completeness by feed and day (beneficiary bank),FAIL,ACH feed loses the beneficiary bank from 2022-09-09 (6.03% of rows on averag...
2,C03,Exact duplicate rows,WARN,"20 exact duplicate rows out of 31,891,251"
3,C04,"Allowed values (currencies, payment formats)",PASS,0 rows outside 15 currencies / 7 formats
4,C05,Non-positive amounts,PASS,0 rows with an amount <= 0
5,C06,Paid = received when the currency is the same,PASS,0 same-currency rows where paid and received differ
6,C07,"Freshness (feed arrival lag, hours after day end)",PASS,"latest feed arrives 3.5 h after day end; the source file also carries 6,987 ..."
7,C08,Daily volume against the same weekday,WARN,"largest deviation +131% on 2022-09-01 (4,465,985 rows)"
8,C09,Self-transfers share per day,FAIL,"45% self-transfers on 2022-09-01 (1,945,611 'Reinvestment' rows, none on oth..."
9,C10,Accounts missing from the account master,PASS,"0 of 2,076,983 accounts seen in the feed are missing from the master"


## 3. Business meaning: flows and formats

In [5]:
t('bm_flow_types')

,flow_type,n_rows,share_rows,laundering_rows,share_of_laundering
0,between banks,28986700,0.908923,29939.0,0.961247
1,self-transfer (same account),2559376,0.080253,60.0,0.001926
2,"same bank, different account",242618,0.007608,247.0,0.007930
3,beneficiary bank missing,102557,0.003216,900.0,0.028896


In [6]:
t('profile_formats')[['payment_format', 'n_rows', 'laundering_rate', 'share_of_laundering']]

,payment_format,n_rows,laundering_rate,share_of_laundering
0,Cheque,12280058,0.000181,0.071277
1,Credit Card,8777816,0.000154,0.043473
2,ACH,3861424,0.006905,0.856065
3,Cash,3217531,0.000207,0.021383
4,Reinvestment,1945611,0.000000,0.000000
5,Wire,1119774,0.000000,0.000000
6,Bitcoin,689037,0.000353,0.007802


## 4. Rule validation: documentation vs code, precision, recall, coverage

In [7]:
t('doc_vs_code', 12).query('gap')[['rule_id', 'name', 'alerts_documented', 'alerts_implemented', 'alerts_in_both', 'tp_accounts_missed_by_code']]

,rule_id,name,alerts_documented,alerts_implemented,alerts_in_both,tp_accounts_missed_by_code
1,R02,Structuring below 10k,668,786,337,0
3,R04,Rapid movement of funds,1067,9877,971,0
9,R10,Cash activity,675,663,143,5


In [8]:
r = t('rule_performance', 24)
r[r.ruleset == 'implemented'][['rule_id', 'name', 'alerts', 'share_of_alerts', 'precision', 'recall', 'unique_tp_accounts']].round(4)

,rule_id,name,alerts,share_of_alerts,precision,recall,unique_tp_accounts
12,R01,Large single transfer,1343,0.0739,0.0030,0.0034,1
13,R02,Structuring below 10k,786,0.0433,0.0076,0.0068,5
14,R03,Fan-out,0,0.0000,NaN,0.0000,0
15,R04,Rapid movement of funds,9877,0.5437,0.0028,0.0319,25
16,R05,Activity spike,47,0.0026,0.0213,0.0011,1
17,R06,Fan-in,83,0.0046,0.0964,0.0091,8
18,R07,High-risk corridor,611,0.0336,0.0016,0.0011,0
19,R08,Crypto / cross-currency,0,0.0000,NaN,0.0000,0
20,R09,Round amounts,3476,0.1914,0.0058,0.0182,10
21,R10,Cash activity,663,0.0365,0.0030,0.0023,2


In [9]:
t('typology_coverage')

,typology,visible_attempts,detected_attempts,detection_rate,rules_designed_for_it
0,FAN-OUT,54,4,0.074074,R03
1,FAN-IN,69,6,0.086957,R06
2,BIPARTITE,70,8,0.114286,"R03, R06"
3,RANDOM,47,6,0.127660,R04
4,GATHER-SCATTER,85,14,0.164706,"R03, R06"
5,SCATTER-GATHER,60,10,0.166667,NaN
6,STACK,94,16,0.170213,R04
7,CYCLE,47,13,0.276596,NaN


## 5. Tuning: ATL/BTL, capacity, and a false signal that must not be escalated

In [10]:
t('tuning_proposal_impact', 12)[['rule_id', 'action', 'alerts_now', 'alerts_after', 'tp_alerts_now', 'tp_alerts_after']]

,rule_id,action,alerts_now,alerts_after,tp_alerts_now,tp_alerts_after
0,R01,retire,1343,0,4.0,0.0
1,R02,fix,786,668,6.0,3.0
2,R03,tune,0,217,0.0,10.0
3,R04,fix,9877,1067,28.0,12.0
4,R05,exclude,47,21,1.0,1.0
5,R06,tune,83,234,8.0,17.0
6,R07,keep,611,611,1.0,1.0
7,R08,keep,0,0,0.0,0.0
8,R09,tune,3476,1655,20.0,7.0
9,R10,fix,663,675,2.0,5.0


In [11]:
t('capacity_weekly')[['wk', 'alert_capacity', 'alerts_implemented', 'alerts_tuned', 'load_implemented', 'load_tuned']].round(2)

,wk,alert_capacity,alerts_implemented,alerts_tuned,load_implemented,load_tuned
0,1,1800,13922,3923,7.73,2.18
1,2,1800,2875,1543,1.60,0.86
2,3,514,1368,809,2.66,1.57


In [12]:
t('payroll_verdict').T

,0
corporate_alerts,114
baseline,75.8
spike,0.503958
payroll_driven_alerts,47
share_known_payroll,1.0
laundering_hits,0.0
escalated,0
payments,379
paying_accounts,26
median_salary_eur,2947.54


## 6. New network scenario R13 (cycles, scatter-gather)

In [13]:
t('network_scenario_performance')

,pattern,alerts,tp_alerts,precision,tp_accounts,tp_accounts_new_vs_tuned_rules,alerts_per_week
0,all R13,48,14.0,0.291667,14,10,21.0000
1,cycle,43,9.0,0.209302,9,5,18.8125
2,scatter-gather,5,5.0,1.000000,5,5,2.1875


In [14]:
t('network_typology_coverage').round(3)

,typology,visible_attempts,rules_now,tuned_rules,r13_alone,tuned_rules_plus_r13
0,BIPARTITE,70,0.114,0.114,0.000,0.114
1,CYCLE,47,0.277,0.213,0.043,0.255
2,FAN-IN,69,0.087,0.101,0.072,0.159
3,FAN-OUT,54,0.074,0.093,0.074,0.130
4,GATHER-SCATTER,85,0.165,0.153,0.059,0.176
5,RANDOM,47,0.128,0.043,0.085,0.128
6,SCATTER-GATHER,60,0.167,0.100,0.183,0.267
7,STACK,94,0.170,0.138,0.053,0.170


## 7. Challenger model: leakage, temporal validation, Simpson's paradox

In [15]:
t('model_metrics')[['version', 'validation', 'roc_auc', 'avg_precision']].round(3)

,version,validation,roc_auc,avg_precision
0,v1 first draft,random split of account-days,0.986,0.568
1,"v1 features, temporal test","train <= 2022-09-10, test after",0.983,0.558
2,v2 challenger,"train <= 2022-09-10, test after",0.942,0.251


In [16]:
t('model_capacity')[['approach', 'alerts_per_day', 'precision', 'recall_accounts']].round(3)

,approach,alerts_per_day,precision,recall_accounts
0,Tuned rules (all alerts),277.5,0.028,0.095
1,Challenger top 257 per day (capacity),257.0,0.155,0.456
2,Tuned rules + challenger (union),NaN,NaN,0.507


In [17]:
t('model_simpson', 12).pivot(index='segment', columns='risk_rating', values='precision').round(3)

risk_rating,all,high,low,medium
segment,,,,
corporate,0.115,0.381,0.00,0.120
retail,0.143,0.543,0.05,0.197
sme,0.214,0.348,0.00,0.152


## 8. Recommendations
See `docs/recommendations_roadmap.md`: fix the 3 code gaps and the ACH feed, exclude known payroll, retune with ATL/BTL evidence, add R13, close KYC gaps, then rank the queue with the challenger score under model governance.

In [18]:
{k: round(v, 3) if isinstance(v, float) else v for k, v in f['tuning'].items()}

{'alerts_now': 18165.0,
 'alerts_after': 6275.0,
 'change_in_alerts': -0.655,
 'precision_now': 0.006,
 'precision_after': 0.015,
 'tp_accounts_now': 94.0,
 'tp_accounts_after': 85.0,
 'tp_accounts_lost': 25.0,
 'tp_accounts_gained': 16.0}